# Higher Education Data ETL Pipeline

In [ ]:
# !pip install sqlalchemy
# !pip install psycopg2

In [1]:
import pandas as pd
import re

# FUNCTION: Extract Year From Filename
def extract_year(file_path):
    filename = file_path.split("/")[-1]

    # Look for any 4-digit year after letters and before _ or .
    match = re.search(r"[a-zA-Z]+(\d{4})(?:_|\.|$)", filename)
    if match:
        return int(match.group(1))

    raise ValueError(f"Could not extract year from filename: {filename}")

# FUNCTION: Read & Filter a CSV in chunks
def load_filtered_chunks(
    file_path,
    usecols,
    string_cols,
    chunksize=100_000
):
    year = extract_year(file_path)

    """
    Reads a CSV in chunks, keeps only selected columns,
    filters rows as needed, adds year of data source capture.
    """
    for chunk in pd.read_csv(
        file_path,
        usecols=usecols,
        dtype = string_cols,
        chunksize=chunksize,
        low_memory=False
    ):
        # Normalize column names to lowercase to match postgresql
        chunk.columns = [col.lower() for col in chunk.columns]
        # 
        # Add year column to each chunk
        chunk["year"] = year

        # Filter rows here if needed
        yield chunk

C:\Users\KDSharp\AppData\Roaming\Python\Python310\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
from sqlalchemy import create_engine

# FUNCTION: Upload a chunk to PostgreSQL
def upload_chunk(df, table_name, engine):
    df.to_sql(
        table_name,
        engine,
        if_exists="append",
        index=False,
        method="multi",
        chunksize=10_000
    )

In [3]:
# FUNCTION: Process a file
# def process_file(file_path, usecols, table_name, engine):
#     for chunk in load_filtered_chunks(file_path, usecols):
#         upload_chunk(chunk, table_name, engine)

def process_file(file_path, usecols, table_name, engine, string_cols):
    for chunk in load_filtered_chunks(file_path, usecols, string_cols):
        upload_chunk(chunk, table_name, engine)

In [4]:
# ESTABLISH THE POSTGRESQL CONNECTION

import glob
from sqlalchemy import create_engine

# PostgreSQL connection parameters
from utility import portnumber, host, database, username, password

# Connection parameters
HOST = host
PORT = portnumber
DBNAME = database
USER = username
PASSWORD = password

# Ensure Server is running and Destination Table exists within the database
engine = create_engine(f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}")


### Integrated Postsecondary Education Data System (IPEDS)<br />

Completions - Awards/degrees conferred by program<br />
https://nces.ed.gov/ipeds/datacenter/DataFiles.aspx

In [ ]:
# PROCESS EACH FILE IN THE DIRECTORY
files = glob.glob("data/education/c20yy_a/*.csv")

for file in files:
    process_file(
        file_path=file,
        usecols= [          # Extract columns of interest to match SQL Table.
            'UNITID',
            'CIPCODE',
            'MAJORNUM',
            'AWLEVEL',
            'CTOTALT'
            ],
        string_cols = {"UNITID": str, "CIPCODE": str, "MAJORNUM": str, "AWLEVEL": str},
        table_name="edu_c20yy_a",
        engine=engine
    )

Completions Dictionary - Awards and Program Definitions<br />
[https://nces.ed.gov/ipeds/datacenter/DataFiles.aspx](https://nces.ed.gov/ipeds/datacenter/DataFiles.aspx?year=2024&surveyNumber=-1&sid=bdee6418-9669-4606-b719-8326debcac04&rtid=7#:~:text=2024-,Completions,-Awards/degrees%20conferred)

In [ ]:
# PROCESS EACH FILE IN THE DIRECTORY
import os

# Extract data definitions from Excel and save to CSV
files = glob.glob("data/education/c20yy_dict/*.xlsx")
sheet = 'Frequencies'
string_cols = ["CodeValue", "CodeValue1"] # Preserve leading '0's

for file in files:
    try:
        year = extract_year(file)
        out_csv = f"data/education/c20yy_dict/c{year}_dict.csv"

        # Skip processing if CSV already exists ---
        if os.path.exists(out_csv):
            print(f"Skipping {file} → {out_csv} already exists")
            continue

        df = pd.read_excel(
            file, 
            sheet_name=sheet,
            converters={col: str for col in string_cols}
            )

        df.to_csv(out_csv, index=False)
        print(f"Saved: {out_csv}")

        # Process new CVS dictionary
        process_file(
            file_path=out_csv,
            usecols= [          # Extract columns of interest to match SQL Table.
                'VarName',
                'VarNumber',
                'CodeValue',
                'ORDINAL_POSITION',
                'VarName1',
                'CodeValue1',
                'ValueLabel',
                'Frequency',
                'Percent'
            ],
            string_cols = {"CodeValue": str, "CodeValue1": str},
            table_name="edu_c20yy_dict",
            engine=engine
        )

    except ValueError:
        print(f"Sheet {sheet} not found in {file}")
    

Saved: data/education/c20yy_dict/c2023_dict.csv
Saved: data/education/c20yy_dict/c2024_dict.csv


Institutional Characteristics<br />
https://nces.ed.gov/ipeds/datacenter/DataFiles.aspx

In [16]:
# PROCESS EACH FILE IN THE DIRECTORY
files = glob.glob("data/education/hd20yy/*.csv")

for file in files:
    process_file(
        file_path=file,
        usecols= [          # Extract columns of interest to match SQL Table.
            'LONGITUD',
            'LATITUDE',
            'IALIAS',
            'ADDR',
            'CITY',
            'STABBR',
            'ZIP',
            'FIPS',
            'SECTOR',
            'ICLEVEL',
            'CONTROL',
            'HLOFFER',
            'UGOFFER',
            'GROFFER',
            'HDEGOFR1',
            'DEGGRANT',
            'COUNTYCD',
            'COUNTYNM',
            'CNGDSTCD',
            'UNITID',
            'INSTNM'
            ],
        string_cols = {"UNITID": str, "COUNTYCD": str, "FIPS": str, "SECTOR": str},
        table_name="edu_hd20yy",
        engine=engine
    )

Classification of Instructional Programs (CIP) Codes<br />
https://nces.ed.gov/ipeds/cipcode/resources.aspx?y=56

In [ ]:
# EXTRACT
edu_cipcodes_2020_df = pd.read_csv("data/education/CIPCode2020.csv")

# TRANSFORM
# Normalize column names to lowercase to match postgresql
edu_cipcodes_2020_df.columns = [col.lower() for col in edu_cipcodes_2020_df.columns]

# Remove Excel formatting
def clean_excel_format(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.startswith('="') and s.endswith('"'):
        s = s[2:-1]
    return s

for col in edu_cipcodes_2020_df.columns:
    edu_cipcodes_2020_df[col] = edu_cipcodes_2020_df[col].map(clean_excel_format)


# LOAD
DESTINATION_TABLE = "edu_cipcodes_2020"

# # Import Cleaned data to Postgresql destination table
edu_cipcodes_2020_df.to_sql(
    DESTINATION_TABLE,
    engine,
    if_exists="append", # or "replace"
    index=False,
    method="multi"      # batches inserts for speed
)